In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

PROJECT_ROOT = Path("/home/nixos/workspace/TradePilot")
DB_PATH = PROJECT_ROOT / "data/tradepilot.duckdb"
LAKEHOUSE = PROJECT_ROOT / "data/lakehouse"
derived_path = LAKEHOUSE / "derived/derived.etf_aw_sleeve_daily/*/*/*.parquet"

con = duckdb.connect(str(DB_PATH), read_only=True)

In [ ]:
derived_files = sorted(LAKEHOUSE.glob("derived/derived.etf_aw_sleeve_daily/*/*/*.parquet"))
len(derived_files), derived_files[:3]

In [ ]:
con.sql(f"""
select
  sleeve_code,
  sleeve_role,
  min(trade_date) as start_date,
  max(trade_date) as end_date,
  count(*) as rows
from read_parquet('{derived_path}')
group by sleeve_code, sleeve_role
order by sleeve_code
""").df()

In [ ]:
plot_df = con.sql(f"""
select trade_date, sleeve_code, adj_close
from read_parquet('{derived_path}')
where trade_date >= date '2025-01-01'
order by trade_date
""").df()

pivot = plot_df.pivot(index="trade_date", columns="sleeve_code", values="adj_close")
pivot.plot(figsize=(12, 5), title="Stage C ETF AW Sleeves - Adj Close")